### 0. Database Connection Setup
Initializes the Ibis connection to the DuckDB database to load the fixed and yearly FAME tables.

In [ ]:
import ibis
import pandas as pd
from utils.f_0_dirs import get_data_dirs

dirs = get_data_dirs(segment="build")
con = ibis.connect(str(dirs.db_path))

fame_yearly = con.table("fame_yearly")
fame_fixed = con.table("fame_fixed")
print("✅ Connected to database.")

### 1. Small Companies Exclusion
Verify that no firm-year entries have fewer than 10 employees or missing employment data.

In [ ]:
invalid_employees = fame_yearly.filter(
    fame_yearly.employees.isnull() | (fame_yearly.employees < 10)
).count().execute()

assert invalid_employees == 0, f"Found {invalid_employees} invalid employee entries."
print("✅ Employee threshold test passed.")

### 2. UK Registered Numbers Only
Verify that non-UK companies (e.g., prefixes '#', 'IE', 'GI') have been successfully dropped.

In [ ]:
uk_prefixes = [
    '',   # England & Wales (pure numbers)
    'NI', # Northern Ireland Company (post-partition)
    'SC', # Scottish Company
    'OC', # Limited Liability Partnership - LLP (England & Wales)
    'SO', # Limited Liability Partnership - LLP (Scotland)
    'NC', # Limited Liability Partnership - LLP (Northern Ireland)
    'LP', # Limited Partnership (England & Wales)
    'SL', # Limited Partnership (Scotland)
    'ZC', # Unregistered Companies (Section 1043) for England & Wales
    'SZ', # Scottish Unregistered Companies (Section 1043)
    'SG', # Scottish Qualifying Partnership
    'CE', # Charitable Incorporated Organisation (England & Wales)
    'CS', # Scottish Charitable Incorporated Organisation
    'R',  # Older Northern Ireland company (no longer issued)
    'IP'  # Industrial and Provident Societies (cooperatives)
]
non_uk_prefixes = [
    'IE', # Ireland
    'JE', # Jersey
    'IM', # Isle of Man
    'GG', # Guernsey
    'GI', # Gibraltar,
    'SE'  # Société Européenne (European Company),
    '#',  # Foreign legal entites that are traded on LSEG
]

non_uk_regex = r'^(IE|JE|IM|GG|GI|SE|#)'

foreign_fixed = fame_fixed.filter(
    fame_fixed.registered_number.re_extract(non_uk_regex, 1).is_not_null()
).count().execute()
foreign_yearly = fame_yearly.filter(
    fame_yearly.registered_number.re_extract(non_uk_regex, 1).is_not_null()
).count().execute()

assert foreign_fixed == 0, f"Found {foreign_fixed} foreign firm records."
assert foreign_yearly == 0, f"Found {foreign_yearly} foreign firm records."
print("✅ UK-only test passed.")

### 3. Firm Age Check
Verify that no firm age is recorded as negative (e.g., replacing -1 with 0). *Note: Runs if column is present.*

In [ ]:
if 'firm_age' in fame_yearly.columns:
    invalid_age = fame_yearly.filter(fame_yearly.firm_age < 0).count().execute()
    assert invalid_age == 0, f"Found {invalid_age} rows with negative firm age."
    print("✅ Firm age test passed.")
else:
    print("⚠️ firm_age column not present in schema. Skipping test.")

### 4. Duplicate Records Check
Verify that each `registered_number` has strictly one record per `year` in the yearly panel.

In [ ]:
duplicates = fame_yearly.group_by(['registered_number', 'year']) \
    .aggregate(row_count=fame_yearly.count()) \
    .filter(ibis._.row_count > 1).count().execute()

assert duplicates == 0, f"Found {duplicates} duplicate firm-year records."
print("✅ Firm-year duplicate test passed.")

### 5. Same Company Name Check
Verify that no two separate `registered_number`s share the exact same `company_name`.

In [ ]:
same_name = fame_fixed.group_by('company_name') \
    .aggregate(id_count=fame_fixed.registered_number.nunique()) \
    .filter(ibis._.id_count > 1).count().execute()

assert same_name == 0, f"Found {same_name} company names mapped to multiple registered numbers."
print("✅ Shared company name test passed.")

### 6. Subsidiaries Check
Verify that no firm in the dataset is a subsidiary of another firm in the dataset (matching `guo` to `company_name`).

In [ ]:
t1 = fame_fixed.alias('t1')
t2 = fame_fixed.alias('t2')

subsidiaries = t1.inner_join(
    t2, t1.guo == t2.company_name
).filter(t1.registered_number != t2.registered_number).count().execute()

assert subsidiaries == 0, f"Found {subsidiaries} firms that are subsidiaries of others in the dataset."
print("✅ Subsidiaries exclusion test passed.")

### 7. Account Balancing Check
Verify the core accounting identity: Total Assets must equal Liabilities + Shareholders' Funds (allowing £2,000 margin of error).

In [ ]:
imbalance = fame_yearly.filter(
    abs(fame_yearly.total_assets - (fame_yearly.liabilities + fame_yearly.shareholders_funds)) > 2000
).count().execute()

assert imbalance == 0, f"Found {imbalance} rows failing the account balancing check."
print("✅ Account balancing test passed.")

### 8. Outlier Validation (Flagged but Kept)
Verify that negative assets are dropped, but extreme values (negative turnover, high turnover/assets) exist and are handled gracefully.

In [ ]:
negative_assets = fame_yearly.filter(fame_yearly.total_assets < 0).count().execute()
assert negative_assets == 0, "Negative total assets were not properly dropped."

negative_turnover = fame_yearly.filter(fame_yearly.turnover < 0).count().execute()
high_ratio = fame_yearly.filter((fame_yearly.turnover / fame_yearly.total_assets) > 50).count().execute()

print("✅ Negative assets properly dropped.")
print(f"ℹ️ Validated retention of negative turnover rows: {negative_turnover}")
print(f"ℹ️ Validated retention of high turnover/assets rows: {high_ratio}")

### 9. Deflation Check (Conceptual)
Verify that the yearly monetary columns are typed correctly as floats, indicating successful execution of the deflation pipeline.

In [ ]:
assert fame_yearly.turnover.type().is_float64(), "Turnover is not correctly typed as a deflated float."
assert fame_yearly.wages.type().is_float64(), "Wages is not correctly typed as a deflated float."

print("✅ Deflation structural check passed.")